# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# Global parameters
L = 0.2                  # m
N = 200
dx = L / N
alpha0 = 8.23e-5         # k0 / (rho c) [m^2/s]
beta = 0.005             # 1/K
T1, T2 = 200.0, 20.0     # robna pogoja [C]
T0 = T2                  # referencna temperatura v k(T)
t_end = 600.0            # stacionarno stanje (pri 120 s odstopanje se nekaj K)
t_cas = [5.0, 20.0, 60.0, 120.0, 600.0]   # casi za izris profilov
x = np.linspace(0, L, N + 1)

# Funkcije

In [ ]:
# Simbolno: stacionarna resitev d/dx (k(T) dT/dx) = 0
xs, Ls, b, dT, Ts0 = sp.symbols("x L beta Delta_T T_0", positive=True)
T_an_sym = Ts0 + (sp.sqrt(1 + (2 * b * dT + b**2 * dT**2) * (1 - xs / Ls)) - 1) / b
k_rel = 1 + b * (T_an_sym - Ts0)                     # k(T) / k0
tok = sp.simplify(k_rel * sp.diff(T_an_sym, xs))     # mora biti konstanten
T_an = sp.lambdify(xs, T_an_sym.subs({Ls: L, b: beta, dT: T1 - T2, Ts0: T0}), "numpy")

In [ ]:
def alfa(T, beta=beta):
    return alpha0 * (1 + beta * (T - T0))

def desna(T, beta=beta):
    # dT_i/dt = [a_{i+1/2}(T_{i+1} - T_i) - a_{i-1/2}(T_i - T_{i-1})] / dx^2
    a_pol = alfa(0.5 * (T[1:] + T[:-1]), beta)
    tok = a_pol * np.diff(T)
    dTdt = np.zeros_like(T)
    dTdt[1:-1] = (tok[1:] - tok[:-1]) / dx**2
    return dTdt

def dt_krit(beta=beta):
    return dx**2 / (2 * alfa(max(T1, T2), beta))

def simuliraj(t_cas, dt, beta=beta, T_zac=None):
    # eksplicitna Eulerjeva shema; vrne profile ob casih t_cas
    T = np.full(N + 1, T2) if T_zac is None else T_zac.copy()
    T[0], T[-1] = T1, T2
    profili, t = [], 0.0
    for t_c in t_cas:
        n_kor = int(round((t_c - t) / dt))
        for _ in range(n_kor):
            T += dt * desna(T, beta)
        t += n_kor * dt
        profili.append(T.copy())
    return np.array(profili)

# Testno območje

In [ ]:
# Pripravljalna naloga 1: limita beta -> 0
display(sp.Eq(sp.Symbol("T(x)"), T_an_sym))
print("limita beta -> 0:", sp.simplify(sp.limit(T_an_sym, b, 0)))
print("k(T) dT/dx =", tok, " (konstanten tok)")
koren = lambda e: e.replace(lambda q: q.is_Pow and q.exp == sp.Rational(1, 2), lambda q: sp.sqrt(sp.factor(sp.expand(q.base))))
print("T(0) =", sp.simplify(koren(T_an_sym.subs(xs, 0))), ",  T(L) =", sp.simplify(koren(T_an_sym.subs(xs, Ls))))

In [ ]:
# Pripravljalna naloga 2: ukrivljenost  T'' = -(k'/k) T'^2  < 0 za beta > 0
d2 = sp.simplify(sp.diff(T_an_sym, xs, 2))
display(sp.Eq(sp.Symbol("T''(x)"), d2))
d2_np = sp.lambdify(xs, d2.subs({Ls: L, b: beta, dT: T1 - T2, Ts0: T0}), "numpy")
print(f"T'' na [0, L]: od {d2_np(x).min():.1f} do {d2_np(x).max():.1f} K/m^2  -> konkaven profil")

# Naloga 1

In [ ]:
print(f"dx = {dx * 1e3:.1f} mm,  alpha_max = {alfa(T1):.3e} m^2/s,  dt_c = {dt_krit() * 1e3:.3f} ms")
T_test = np.full(N + 1, T2); T_test[0] = T1
print("dT/dt ob t = 0 ob vrocem robu:", np.round(desna(T_test)[:4], 2), "K/s")

# Naloga 2

In [ ]:
dt = 0.4 * dt_krit()
profili = simuliraj(t_cas, dt)
T_lin = T1 - (T1 - T2) * x / L

fig = plt.figure(figsize=(14, 5))
ax = fig.add_subplot(121)
for t_c, T in zip(t_cas, profili):
    ax.plot(x * 100, T, label=f"$t = {t_c:.0f}$ s")
ax.set_xlabel("$x$ [cm]", fontsize=14)
ax.set_ylabel(r"$T$ [$^\circ$C]", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)

ax = fig.add_subplot(122)
ax.plot(x * 100, profili[t_cas.index(t_end)], label=f"numerično ($t = {t_end:.0f}$ s)")
ax.plot(x * 100, profili[t_cas.index(120.0)], ":", label="numerično ($t = 120$ s)")
ax.plot(x * 100, T_an(x), "--", color="black", label="analitično, nelinearno")
ax.plot(x * 100, T_lin, "-.", color="gray", label=r"linearno ($\beta = 0$)")
ax.set_xlabel("$x$ [cm]", fontsize=14)
ax.set_ylabel(r"$T$ [$^\circ$C]", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for t_c, T in zip(t_cas, profili):
    print(f"t = {t_c:5.0f} s: max |T_num - T_an| = {np.max(np.abs(T - T_an(x))):.3f} K")
print(f"max |T_an - T_lin| = {np.max(np.abs(T_an(x) - T_lin)):.2f} K")

# Naloga 3

In [ ]:
# 20000 korakov iz hladnega zacetnega stanja; fizikalna resitev mora ostati v [T2, T1]
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(111)
for f in [0.9, 1.0, 1.05, 1.1, 1.2]:
    dt_f = f * dt_krit()
    with np.errstate(over="ignore", invalid="ignore"):
        T = simuliraj([20000 * dt_f], dt_f)[0]
    ok = np.all((T >= T2 - 1e-9) & (T <= T1 + 1e-9))
    print(f"dt = {f:.2f} dt_c: T v [{np.nanmin(T):7.1f}, {np.nanmax(T):7.1f}] C  -> {'stabilno' if ok else 'NESTABILNO (T izven [T2, T1])'}")
    ax.plot(x * 100, T, label=rf"$\Delta t = {f}\,\Delta t_c$")
ax.set_ylim(-100, 400)
ax.set_xlim(0, 5)
ax.set_xlabel("$x$ [cm]", fontsize=14)
ax.set_ylabel(r"$T$ [$^\circ$C]", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

# Razmislek 1

In [ ]:
# Stacionarno: alpha(T) T'' + alpha' (T')^2 = 0 -> clena se natancno izravnata
Tx = np.gradient(T_an(x), x)
Txx = d2_np(x)
c1 = alfa(T_an(x)) * Txx
c2 = alpha0 * beta * Tx**2

fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(111)
ax.plot(x * 100, c1, label=r"$\alpha(T)\,T''$")
ax.plot(x * 100, c2, label=r"$\alpha_0 \beta\,(T')^2$")
ax.plot(x * 100, c1 + c2, "--", color="black", label="vsota")
ax.set_xlabel("$x$ [cm]", fontsize=14)
ax.set_ylabel("[K/s]", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

# Razmislek 2

In [ ]:
# Superpozicija ne velja: odziv na 2 Delta_T ni 2-kratnik odziva na Delta_T
th = lambda d: (T_an_sym - Ts0).subs({Ls: L, b: beta, dT: d, xs: L / 2})
print(f"theta(L/2) pri dT = 90:  {float(th(90)):.3f} K")
print(f"theta(L/2) pri dT = 180: {float(th(180)):.3f} K   (2 x {float(th(90)):.3f} = {2 * float(th(90)):.3f} K)")